In [1]:
import cv2
import os
import cv2
import json
import numpy as np
import xml.etree.ElementTree as ET
from ultralytics import YOLO
from tqdm import tqdm
import math
from pathlib import Path
import matplotlib.pyplot as plt


In [2]:

def pick_file_with_priority(base_dir, pattern):
    """
    Priority:
    1. calibrated
    2. raw
    3. anything else
    """
    if not base_dir.exists():
        return None

    all_files = list(base_dir.rglob(pattern))
    if not all_files:
        return None

    calibrated = [f for f in all_files if "calibrated" in str(f).lower()]
    raw = [f for f in all_files if "raw" in str(f).lower()]

    if calibrated:
        return calibrated[0]
    if raw:
        return raw[0]
    return all_files[0]


def resolve_datapoint_files(root_dir):
    root = Path(root_dir)

    # -------------------------
    # PNG (browse)
    # -------------------------
    browse_dir = root / "browse"
    png_file = pick_file_with_priority(browse_dir, "*.png")

    # -------------------------
    # XML (data)
    # -------------------------
    data_dir = root / "data"
    xml_file = pick_file_with_priority(data_dir, "*.xml")

    return {
        "png": str(png_file) if png_file else None,
        "xml": str(xml_file) if xml_file else None
    }

In [3]:
data_path = r"C:\Users\dhrit\Downloads\ch2_ohr_nrp_20210405T0245288072_d_img_d32"

In [4]:
files = resolve_datapoint_files(data_path)

print(files["png"])
print(files["xml"])

C:\Users\dhrit\Downloads\ch2_ohr_nrp_20210405T0245288072_d_img_d32\browse\raw\20210405\ch2_ohr_nrp_20210405T0245288072_b_brw_d32.png
C:\Users\dhrit\Downloads\ch2_ohr_nrp_20210405T0245288072_d_img_d32\data\raw\20210405\ch2_ohr_nrp_20210405T0245288072_d_img_d32.xml


In [5]:
model=r"C:\Users\dhrit\OneDrive\Desktop\crater_depth\best_final.pt"

In [6]:


model = YOLO(model) 

In [7]:
CRATER_LABELS = {
    0: "crater_small",
    1: "crater_medium",
    2: "crater_large"
}

In [8]:
def compute_diameter(bbox, pixel_res):
    x1, y1, x2, y2 = bbox

    width_px = x2 - x1
    height_px = y2 - y1

    diameter_px = max(width_px, height_px)
    diameter_m = diameter_px * pixel_res

    return diameter_m

In [9]:

def visualize_and_save(image, boxes, output_path):

    img_boxes = image.copy()

    for i, obj in enumerate(boxes):
        x1, y1, x2, y2 = obj["bbox"]

        # draw rectangle
        cv2.rectangle(img_boxes, (x1, y1), (x2, y2), (255, 0, 0), 2)

        # compute center of bbox
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        # put crater_id at center
        cv2.putText(
            img_boxes,
            f"{i}",
            (cx, cy),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    # convert BGR → RGB for matplotlib
    left = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    right = cv2.cvtColor(img_boxes, cv2.COLOR_BGR2RGB)

    fig, axs = plt.subplots(1, 2, figsize=(15, 8))

    axs[0].imshow(left)
    axs[0].set_title("Raw Image")
    axs[0].axis("off")

    axs[1].imshow(right)
    axs[1].set_title("YOLO Craters (ID centered)")
    axs[1].axis("off")

    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

In [10]:
def random_crop_640(image):
    h, w = image.shape[:2]

    if h < 640 or w < 640:
        raise ValueError("Image smaller than 640x640")

    x = np.random.randint(0, w - 640)
    y = np.random.randint(0, h - 640)

    crop = image[y:y+640, x:x+640]
    return crop, (x, y)

In [11]:
def classify_depth(depth_m):
    if depth_m is None:
        return "unknown"

    if depth_m < 0.3:
        return "very_shallow"
    elif depth_m < 0.8:
        return "shallow"
    elif depth_m < 1.5:
        return "medium"
    else:
        return "deep"

In [12]:
def is_valid_sun_metadata(meta):
    return (
        meta is not None and
        meta.get("sun_elevation") is not None and
        meta.get("sun_azimuth") is not None
    )

In [13]:

# 1. Metadata parser

def parse_pds_metadata(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    ns = {"isda": "https://isda.issdc.gov.in/pds4/isda/v1"}

    def get(tag):
        node = root.find(f".//isda:{tag}", ns)

        if node is None or node.text is None:
            return None

        try:
            return float(node.text)
        except ValueError:
            return None

    return {
        "pixel_resolution": get("pixel_resolution"),
        "sun_elevation": get("sun_elevation"),
        "sun_azimuth": get("sun_azimuth"),
    }

#pixel_resolution = 0.25  # m/px

# 2. Rotate image

def rotate_image(img, angle_deg):
    h, w = img.shape[:2]
    center = (w // 2, h // 2)

    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    return cv2.warpAffine(img, M, (w, h))



# 3. Depth estimation

def estimate_depth_directional(gray, bbox, pixel_res, sun_elev, sun_azimuth):

    x1, y1, x2, y2 = bbox
    crater = gray[y1:y2, x1:x2]

    crater = cv2.GaussianBlur(crater, (5, 5), 0)

    # rotate so sun direction aligns with x-axis
    rotated = rotate_image(crater, -sun_azimuth)

    # shadow detection
    thresh = np.percentile(rotated, 20)
    shadow = rotated < thresh

    coords = np.column_stack(np.where(shadow))

    if len(coords) < 20:
        return None

    shadow_px = coords[:, 1].max() - coords[:, 1].min()
    shadow_m = shadow_px * pixel_res

    depth = shadow_m * math.tan(math.radians(sun_elev))

    return {
        "shadow_length_px": float(shadow_px),
        "shadow_length_m": float(shadow_m),
        "depth_m": float(depth)
    }



# 4. YOLO + Depth pipeline

def run_pipeline(img_path, xml_path, weights_path, output_json="result.json"):

    model = YOLO(weights_path)

    # Safety checks 
    if not os.path.exists(img_path):
        print(f"Missing image: {img_path}")
        return

    if not os.path.exists(xml_path):
        print(f"Missing xml: {xml_path}")
        return

    # Load image 
    image_full = cv2.imread(img_path)
    if image_full is None:
        print(f"Failed to read image: {img_path}")
        return

    image, _ = random_crop_640(image_full)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Metadata 
    meta = parse_pds_metadata(xml_path)

    sun_available = is_valid_sun_metadata(meta)

    if not sun_available:
        print("[WARNING] Sun metadata incomplete, depth estimation will be skipped.")

    # YOLO inference 
    yolo_results = model.predict(image, conf=0.25, verbose=False)

    boxes = []

    for r in yolo_results:
        xyxy = r.boxes.xyxy.cpu().numpy()
        cls = r.boxes.cls.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()

        for i in range(len(xyxy)):
            x1, y1, x2, y2 = map(int, xyxy[i])

            boxes.append({
                "bbox": [x1, y1, x2, y2],
                "class_id": int(cls[i]),
                "label": CRATER_LABELS.get(int(cls[i]), "unknown"),
                "confidence": float(conf[i])
            })

    # Depth estimation AND Classification
    image_results = []

    for i, obj in enumerate(boxes):

        
        if sun_available:
            depth = estimate_depth_directional(
                gray,
                obj["bbox"],
                meta["pixel_resolution"],
                meta["sun_elevation"],
                meta["sun_azimuth"]
            )
        else:
            depth = None

        D = compute_diameter(obj["bbox"], meta["pixel_resolution"])

        if depth is not None:
            d = depth["depth_m"]
            d_D = d / D if D > 0 else None
        else:
            d = None
            d_D = None

        
        if sun_available and d is not None:
            depth_class = classify_depth(d)
        else:
            depth_class = "not_computable"

        image_results.append({
            "crater_id": i,
            "bbox": obj["bbox"],
            "label": obj["label"],
            "class_id": obj["class_id"],
            "confidence": obj["confidence"],
            "depth_m": d,
            "depth_class": depth_class,
            "diameter_m": D,
            "d_by_D": d_D
        })

    # Visualization 
    vis_output_path = os.path.join(
        os.getcwd(),
        os.path.basename(img_path).replace(".png", "_vis.png")
    )

    visualize_and_save(image, boxes, vis_output_path)

    # Save output 
    output = {
        "image": os.path.basename(img_path),
        "craters": image_results
    }

    with open(output_json, "w") as f:
        json.dump(output, f, indent=2)

    print(f"\nSaved results to {output_json}")

In [14]:
if __name__ == "__main__":
    image_dir = files["png"]
    xml_dir   = files["xml"]

    weights   = model

    run_pipeline(image_dir, xml_dir, weights, "results.json")
    #print("ch2_ohr_nrp_20210405T0245288072_d_img_d32.xml")


Saved results to results.json
